# Analysing scanning electron diffraction data using HyperSpy and pyXem

This Noteboot aims to teach the participants the basics on how to use HyperSpy to analyse large scanning electron diffraction (SED) datasets, assuming no prior experience with Python.

Content:

- Loading SED datasets using HyperSpy
- Plotting and visualization
- Difference between navigation and signal dimensions, and the `transpose` functionality
- Working with datasets larger than the available computer memory, via `LazySignals`
- Cropping of signals using the slicing syntax, `inav` and `isig`
- Data reduction via slicing and rebinning.

## Requirements

- HyperSpy 1.7.1 or higher
- pyXem 0.14.2 or higher

## Author

- 2022/06/22: Magnus Nord, TEM-Gemini, Department of Physics, NTNU. The Research Council of Norway is acknowledged for partially funding the creation of this Jupyter Notebook via the InCoMa project (Project number: 315475). Partial funding was also received from ESTEEM3 via the European Union's Horizon 2020 grant number 823717.

## Loading the data

First we need to set a plotting library

In [ ]:
%matplotlib qt5

Then we need to import the HyperSpy library

In [ ]:
import hyperspy.api as hs

### Loading

Now, we're going to look a small SPED dataset. To do this, we use HyperSpy's load method. `load` returns a signal, which must be assigned to a variable.

In [ ]:
s = hs.load("datasets/cpu_sample_sped_small.hspy")

Now lets have a look at this dataset, lets see what `s` is

In [ ]:
s

Here we see that we have a ElectronDiffraction2D `(20, 20|256, 256)`. The numbers to the left of the `|` is the navigation dimension, while the numbers to the right of the `|` is the signal dimensions. The easiest way to show this, is by visualizing the data through the `plot` function in the signal `s`.

This is a small dataset, which has been cropped from a bigger one. We'll look at the full dataset a bit later.

**Note:** the plotting window might be hidden behind the notebook window.

In [ ]:
s.plot()

Here we get two separate windows: one showing the navigation dimension, and one showing the signal dimensions. We see that the signal dimension has the central beam in the middle, but we can't really see much else.

The navigation dimension has a *navigator*, show here with the red box. It can be moved either by using the **left, right, up or down keys on your keyboard**, or we can drag it using the mouse. This enables us to see what the diffraction pattern look like in the different parts of the sample.

One trick here, is to use the `l` key on your keyboard. This changes the intensity in the diffraction image to a log-log plot.

Now we can see the weaker intensity and tell what type of crystal structure we have in the sample.

**Note** the white pixels is due to those pixels having the value zero, which doesn't work for log plot. The solution to this is using a different `norm`. To do this, press the `h` key while having the diffraction window active. Now, change "Norm" from `log` to `symlog`.

## Transposing the data

For me, one of the most powerful aspects of HyperSpy, is the concept of signal and navigation axes. This includes the ability change these around easily. I think the easiest is by showing it.

To do this, we use the `transpose` function. Where we use the shortcut `T` to simply switch the navigation and signal axes.

In [ ]:
st = s.T

In [ ]:
st

So now we've moved the 256 x 256 detector dimensions to the left of the `|`, and the probe positions to the right of the `|`. Lets plot this, and see how it looks like!

Here, use the `+` key on your keyboard with the navigation plot selected to make the navigator easier to select.

In [ ]:
st.plot()

The ability to switch around which dimension is navigation and signal is really powerful!

Tips: use the `+` key to increase the size of the navigator, note that this does not mean it averages over several spectra. It is just a convenience to make it easier to select.

## Loading big data using lazy loading

### Supported file formats

One of the really big challenges with analysing data using open source software like HyperSpy, is getting access to the actual data and metadata. So we have put in a great deal of work in implementing file readers for the most common file formats used in TEM.

Example of this is `dm3`, `dm4`, `emi/ser` and `EMD`. See the [HyperSpy supported formats](https://hyperspy.org/rosettasciio/supported_formats/index.html#supported-formats) for a full list.

In addition, you can turn a NumPy array into a HyperSpy signal.

The file we just loaded is the `hspy` file format, which is a HyperSpy type `HDF5` file.

For the next file we'll be looking at, we'll use the `zspy` format. Which is optimized for big datasets. Note the use of `lazy=True`.

In [ ]:
s1 = hs.load("datasets/cpu_sample_sped.zspy", lazy=True)

In [ ]:
print(s1)

Here we see that the file is `LazyElectronDiffraction2D`, which is similar to the `ElectronDiffraction2D` we saw earlier. But the lazy part means we haven't actually loaded the dataset into memory.

In [ ]:
# Use the "nbytes" attribute of the numpy array to calculate the size on disk
print(s1.data.nbytes / 1e9)

That is about 8.6 GB of data, which could load into memory and process the standard way if you have about 16 GB of RAM. Many of you probably have computers which could do this, but these datasets can get way bigger. All the way up to terabytes. So we need to know how to process this type of data, without loading it all into memory.

If you want to try this on a much bigger dataset after the workshop, you can check out this [Zenodo deposit](https://zenodo.org/record/4312960), specifically the [largest file](https://zenodo.org/record/4312960/files/fe60al40_stripe_pattern.hspy?download=1), which is a magnetic [STEM-DPC](https://en.wikipedia.org/wiki/Scanning_transmission_electron_microscopy#Differential_phase_contrast) dataset.

### Lazy plotting

Plotting is pretty much the same as non-lazy signals.

In [ ]:
s1.plot()

We can also easily do the transpose trick. Note that this requires the file to be **chunked** in a specific way, which I might cover later if there is time.

In [ ]:
s1.T.plot()

Now, lets go through some standard data processing you would do on this type of dataset. Like plotting the diffraction patterns from different regions.

First we need to calibrate the data!

### Calibrating the data

Here we use the added calibrate function. First, lets do the diffraction dimensions. Change the line to get the distance between two diffraction spots, then type a number in `New Length` and `mrad` in `units`.

Note: you can select different probe positions by moving the navigator. For example using the crystalline silicon in the top left.

(Getting the correct calibration here is not important, so just input any number)

In [ ]:
s1.calibrate()

Then, we calibrate the navigation dimension through a little "trick": by transposing the signal.

In [ ]:
s1T = s1.T

Then running `calibrate` on this new, transposed, signal. Here, we see that the diffraction dimension is properly calibrated, as the calibration is retained when transposing.

We use the knowledge that the distance between the outer parts of the four bottom-right features is 2.1 um.

In [ ]:
s1T.calibrate()

Then, transpose the signal back again. To a new calibrated signal: `s_cal` 

In [ ]:
s_cal = s1T.T

In [ ]:
s_cal.plot()

The calibration can also be set directly, if you know the scaling.

In [ ]:
s_cal.axes_manager.gui()

### Getting the diffraction patterns

Now, lets extract some diffraction patterns from different regions.

For this, we'll grab a couple of probe positions, and take the average. For this, we should learn about slicing!

#### Interlude: how to slice

First, lets get the top left position in the dataset, where the crystalline Si is. To do this, we'll use the `inav` function, which we can use to crop the data in the navigation dimensions.

* Need to use square brackets `[]`
* Python is `zero indexed`, meaning the first index is 0.

In [ ]:
print(s_cal.inav[0, 0])

In [ ]:
s_cal.inav[0, 0].plot(norm='symlog')

As we can see, this works nicely on lazy data. But this is only a single position! We want to average a couple of patterns, to get a nicer signal.

Now, lets first have a look at the big dataset, to see which positions we want to get. We use the red, top left, numbers in the signal plot to see which indices we want to get.

In [ ]:
s_cal.plot()

Lets grab the first 10 x 10 pixels, via `inav`. Here we use the `:` to get the indices between a number of values.

In [ ]:
s_cal.inav[0:10, 0:10]

In [ ]:
s_cal.inav[0:10, 0:10].plot(norm="symlog")

Now we can use "command chaining" to get the mean of these 100 signals.

In [ ]:
s_si = s_cal.inav[0:10, 0:10].mean()

This gives us a nice diffraction pattern, with just the crystalline Si!

In [ ]:
s_si.plot()

#### Diffraction from other structures

Lets grab diffraction patterns from some of the other structures. Again, we use the full dataset to find the indices, then `inav` to get the specific patterns.

In [ ]:
s_cal.plot()

In [ ]:
s_cryst = s_cal.inav[106:117, 89:103].mean()

In [ ]:
s_amorphous = s_cal.inav[143:162, 75:95].mean()

### Computing the data

This far we haven't actually loaded the data into memory, all these diffraction patterns are still lazy.

Looking at the `data` attribute in the signal, we see that it is a dask object. I'll get back to this a bit later.

In [ ]:
print(s_si.data)

We have just "chained" a number of operations together, without actually executing them. To do this, we need to use `compute`.

In [ ]:
s_si.compute()

Lets have a look at `data` again.

In [ ]:
s_si.data

Now we get a bunch a numbers, which means we've loaded the slice of data into memory, and gotten the mean of the 100 diffraction patterns.

In [ ]:
s_si

We also see that the signal has become a `ElectronDiffraction2D`, not a `LazyElectronDiffraction2D`.

Do this for the rest of the signals.

In [ ]:
s_cryst.compute()

In [ ]:
s_amorphous.compute()

### Saving the patterns

Having done this processing, it is nice to store the results. This is done via the `save` function. It can save in a number of formats, as shown in the `Write` column in [HyperSpy supported formats](https://hyperspy.org/hyperspy-doc/current/user_guide/io.html#supported-formats).

The default is the HyperSpy format `.hspy`, which will retain all the metadata and calibration.

In [ ]:
s_si.save("si_substrate.hspy")

#### `.tif` file format

If you want to open the data in other scientific software, I would use `.tif`. However, here we need to keep in mind the data type of data. To see this, we have a look at `data.dtype`

In [ ]:
s_si.data.dtype

`float64` is a 64-bit floating point number, which most software does not support. If you want to open this is in ImageJ, you can use 16-bit integers.

In [ ]:
s_si.change_dtype('int16')

In [ ]:
s_si.save("si_substrate.tf")

On no! We got an error! Specifically a `ValueError`. If you get errors, go all the way to the bottom of the error, and read the text. Most of the time, this error should tell you how to resolve the issue. Specifically it tells us the `tf` file format is not supported, since it is, in fact, due to a type. In addition, it tells us exactly how to solve the issue by showing all the formats which *is* supported.

Note that there are also `Warnings`, which can look similar to errors. However, they will say "Warning", and will give you information about something. Typically, this will not prevent you from doing whatever you want to do. Errors means whatever you were trying to do did NOT work, so you need to figure out how to fix it.

* `Errors`: means whatever you were trying to do did not work. Go to the bottom of the error, and try to understand why you get it. Googling the error message is a possible way.
* `Warnings`: typically, this means whatever you try to do worked. However, there might be something which can be optimized or something not working as expected.

So lets save it with the correct file ending.

In [ ]:
s_si.save("si_substrate.tif")

While this works, we would rather visualize the diffraction in a nicer way.

### Making a nicer plot

So we can save the diffraction images as individual jpg or tif images, but that is not what we really want. We want to plot them side by side, with a scalebar and some kind of title.

First, lets remake the Si signal, since we changed the `dtype` earlier.

In [ ]:
s_si = s_cal.inav[0:10, 0:10].mean()

In [ ]:
s_si.compute()

HyperSpy has a number of plotting functions in `hs.plot`. One of them is `plot_images`.

In [ ]:
hs.plot.plot_images((s_si, s_cryst, s_amorphous))

This shows the 3 images, but not really the way we want.

- We can't see the weaker reflections, which are the ones we really care about. We can solve this by plotting them as log, as we've done this far.
- The scalebar is missing
- We don't want the numbers on the sides of the plot
- The colorbar is not really useful here, so we should remove them
- Some text explaining what we're looking at would be nice
- Also: we want to correct for the dead pixels

#### Fixing dead pixels

Lets start with the dead pixels. We can sometimes see them in individual patterns, but it is easier to get the mean of the whole dataset.

In [ ]:
s_mean = s_cal.mean()

This is a lazy signal, so we need to use `compute`. This will load the data into memory in small chunks, do the mean, and release the data. Ergo, it will _not_ load the full dataset into memory at the same time. Thus, even if the dataset is larger than your available memory, it will still be able to compute the mean.

In [ ]:
s_mean.compute()

Now, plot it using `norm=symlog`, and have a look at the dead pixels.

In [ ]:
s_mean.plot(norm="symlog")

So two of them are actually not 0, but has almost zero value. We could just have fixed these manually, but lets do this in more clever way via thresholding.

We can do many operations directly on the HyperSpy signals, which will create a new signal. Here we use the "smaller than" operator.

In [ ]:
s_dead_pixels = s_mean < 0.1

In [ ]:
s_dead_pixels

This created a new signal, with the same type and size as `s_mean`.

Plotting it, we see that it has boolean values, and it has indeed found the 3 dead pixels!

In [ ]:
s_dead_pixels.plot()

Knowing where the dead pixels are, we now fix the dead pixels using an in-built function.

It works by setting the intensity of the dead pixel, to the average of its neighbors.

In [ ]:
s_si_corr = s_si.correct_bad_pixels(s_dead_pixels, lazy_output=False, inplace=False)

In [ ]:
s_si_corr.plot()

Then, apply this to the two other signals.

In [ ]:
s_cryst_corr = s_cryst.correct_bad_pixels(s_dead_pixels, lazy_output=False, inplace=False)

In [ ]:
s_amorphous_corr = s_amorphous.correct_bad_pixels(s_dead_pixels, lazy_output=False, inplace=False)

#### Making plots log-log

With the dead pixels fixed, lets make the signal values logarithmic, so we can easily see the weaker reflections.

For this, we'll use `numpy`'s `log` function, which can operate directly on the signals. `numpy` is a widely used scientific python library for working with multidimensional arrays, and HyperSpy heavily relies on `numpy`.

In [ ]:
import numpy as np

Many of numpy's functions can operate directly on the HyperSpy signals.

In [ ]:
s_si_log = np.log(s_si_corr)

In [ ]:
s_si_log.plot()

Then do this on the two other signals.

In [ ]:
s_cryst_log = np.log(s_cryst_corr)

In [ ]:
s_cryst_log.plot()

In [ ]:
s_amorphous_log = np.log(s_amorphous_corr)

In [ ]:
s_amorphous_log.plot()

#### Cropping the diffraction patterns

Since the interesting parts of the pattern is in the middle, we crop out this part.

For this, we use the analog to the `inav`, which is `isig`. This works on the signal dimensions. In addition, we use negative indices crop from the "other" side.

In [ ]:
s_si_log_crop = s_si_log.isig[40:-40, 40:-40]

In [ ]:
s_si_log_crop.plot()

Now, lets apply this to the other signals. However, lets make a slicing object, which can be used for both signals.

In [ ]:
diff_slice = np.s_[40:-40, 40:-40]

In [ ]:
diff_slice

This `diff_slice` object can be used directly for both `inav` and `isig`.

In [ ]:
s_cryst_log_crop = s_cryst_log.isig[diff_slice]

In [ ]:
s_amorphous_log_crop = s_amorphous_log.isig[diff_slice]

Then we visualize this using `plot_images`

In [ ]:
hs.plot.plot_images((s_si_log_crop, s_cryst_log_crop, s_amorphous_log_crop))

#### Customizing plot

This looks better, but we're still lacking a number of things:

- The scalebar is missing
- We don't want the numbers on the sides of the plot
- The colorbar is not really useful here, so we should remove them
- Some text explaining what we're looking at would be nice

For this, we need to have a look at the `docstring` for `plot_images`. Either place a `?` after the function, or click the function, and press `Shift + Tab`

In [ ]:
hs.plot.plot_images

In [ ]:
hs.plot.plot_images(
    (s_si_log_crop, s_cryst_log_crop, s_amorphous_log_crop),
    axes_decor='off',
    colorbar=None,
    suptitle="Comparing diffraction patterns",
    label=["Si", "Polycrystalline", "Amorphous"],
    scalebar='all')

## Getting virtual dark field images

Another common processing we want to do, is to do virtual dark field imaging. Basically to see where the diffraction intensity originates from.

First, lets do this interactively. To do this, we first need to define what type of "aperature" we want to use.

For this, we use the region of interest functionality from HyperSpy. Press `TAB` to get the full list of `roi`'s. Here, a circle is the most relevant.

Then use `Shift + TAB` to see which parameters we should use.

In [ ]:
hs.roi.

In [ ]:
hs.roi.CircleROI

In [ ]:
hs.roi.CircleROI(cx=1, cy=1, r=2)

Then store this a variable.

In [ ]:
circ_roi = hs.roi.CircleROI(cx=1, cy=1, r=0.5)

We use this `roi` with the `plot_integrated_intensity` function. Be careful about making the circle too big, the bigger it is, the more data has to be loaded into memory. This can slow things down, depending on your computer hardware.

In [ ]:
s_cal.plot_integrated_intensity(circ_roi)

#### Getting different positions

Now we want to get this as individual signals, so we can plot it using `plot_images`. The `roi`s can be used with `isig` and `inav`, but also by the function `get_integrated_intensity`.

The trick here, is to get the information about the `CircleROI` while using the live imaging. So move the circle to a location where you want to make a signal, then run `circ_roi` in a cell to get the position.

Change the `cx`, `cy` and `r` values to match the location you want to get a signal.

In [ ]:
circ_roi

In [ ]:
circ0 = hs.roi.CircleROI(cx=15.39, cy=36.8111, r=1.97574, r_inner=0)

In [ ]:
circ1 = hs.roi.CircleROI(cx=27.4524, cy=12.2704, r=1.97574, r_inner=0)

In [ ]:
circ2 = hs.roi.CircleROI(cx=32.2358, cy=33.4836, r=1.97574, r_inner=0)

Then use the `roi`s we've made, to generate the virtual darkfield images.

In [ ]:
s_vdf0 = s_cal.get_integrated_intensity(circ0)

In [ ]:
s_vdf1 = s_cal.get_integrated_intensity(circ1)

In [ ]:
s_vdf2 = s_cal.get_integrated_intensity(circ2)

These are all `lazy`, so use compute to load them into memory.

In [ ]:
s_vdf0

In [ ]:
s_vdf0.compute()

In [ ]:
s_vdf1.compute()

In [ ]:
s_vdf2.compute()

#### Plotting nice virtual dark field

Then, we use `plot_images` to make a nice plot.

In [ ]:
hs.plot.plot_images(
    (s_vdf0, s_vdf1, s_vdf2),
    axes_decor='off',
    colorbar=None,
    suptitle="Virtual dark field")

## Data reduction: how to explore really big datasets

This far, we've used the `lazy` functionality to look at the data. This means we never load the full dataset into memory at the same time. However, sometimes we want to load the full data into memory.

One way is to get lots of memory for your computer, but at some point, the datasets simply become way too big. In those cases, we need to reduce the size of the data somehow.

### Quick and easy

Assuming we want to see the full field-of-view, the easiest way is to simply grab every "N" probe position, or "M" detector pixel.

This is done via `inav` or `isig`.

In [ ]:
s1 = s_cal.inav[::4, ::4].isig[::2, ::2]

In [ ]:
s1

The dataset is now about 128 MiB, and we can easily fit it into memory.

In [ ]:
s1.compute()

In [ ]:
s1.plot()

### Binning

Picking out a subset of the probe positions or diffraction pixels is nice, but a bit "wasteful". We're simply ignoring large parts of the dataset.

A less "wasteful" way, is to sum neighboring pixels. This is done using `rebin`.

In [ ]:
s_rebin = s_cal.rebin(scale=(4, 4, 1, 1), rechunk=False)

In [ ]:
s_rebin

This returns a new signal, with a datatype of unsigned 64-bit integers. This means the maximum value in each detector pixel is:

In [ ]:
2**64

This is quite high! The dataset is acquired in 12-bit, meaning the maximum value after the rebin is:

In [ ]:
2**12 * 4 * 4

Thus, using 64-bit is unnecessary, especially since it requires us to use way more memory than is needed. We could for example use 16-bit integers, which has a maximum value of:

In [ ]:
2**16

We set this using `dtype=np.uint16`.

In [ ]:
s_rebin = s_cal.rebin(scale=(4, 4, 1, 1), dtype=np.uint16, rechunk=False)

In [ ]:
s_rebin

Comparing this to the original `s_rebin`, this dataset is 4 times as small, without losing anything!

In [ ]:
s_rebin.compute()

Now, the signal-to-noise is much better compared to the "every-N-probe-position" method.

In [ ]:
s_rebin.plot()

In [ ]:
s_rebin.T.plot()